In [1]:
import pandas as pd
import numpy as np
import requests

In [2]:
import pandas as pd
import numpy as np
import os, sys

PROJECT_ROOT = os.path.join(os.getcwd(), '..',)#os.path.dirname(os.path.dirname(__file__))
sys.path.insert(1, PROJECT_ROOT) 
from prepare_data import ExchangeData

class BinanceLoader(ExchangeData):

    def __init__(self, exchange, symbol_type):
        super().__init__(exchange, symbol_type)


    def get_ts13(self, datetime:str, timezone:str='Asia/Taipei'):
        ts13 = int(pd.to_datetime(datetime).tz_localize(timezone).timestamp()*1000)
        return ts13
  
    
    def fetch_ohlcv(self, url, symbol, start_ts13, freq, limit, columns, need_col=["datetime", "open", "high", "low", "close", "volume"]):
        """
        Mind
        - timezone = 'UTC'
        - spot: https://binance-docs.github.io/apidocs/spot/en/#kline-candlestick-data
        - usd: https://binance-docs.github.io/apidocs/futures/en/#kline-candlestick-data
        - coin: https://binance-docs.github.io/apidocs/delivery/en/#kline-candlestick-data
        """

        params = {
            'symbol': symbol, 
            'interval': freq, 
            'startTime': start_ts13, 
            'limit':limit,
        }

        data = requests.get(url, params=params).json()
        df = pd.DataFrame(data, columns=columns)
        df['datetime'] = pd.to_datetime(df['open_time'], unit='ms')
        
        return df[need_col].set_index(['datetime'])



In [3]:
bl = BinanceLoader(exchange='binance', symbol_type='usd')
# binance_loader.get_limit_kline()
# binance_loader.get_columns_kline()

In [4]:
df = bl.fetch_ohlcv(
    url = bl.get_end_point() + bl.get_suffix_kline(),
    symbol = 'BTCUSDT',
    start_ts13 = bl.get_ts13('2024-1-11'),
    freq = '1m',
    limit = bl.get_limit_kline(),
    columns = bl.get_columns_kline(),
)

In [ ]:
fetch_ohlcv(self, url, symbol, start_ts13, freq, limit, columns, need_col=["datetime", "open", "high", "low", "close", "volume"]):

In [ ]:
def download_ohlcv(self, start:str, freq:str, symbol_li:list=None,):
    """
    Download ohlcv for the given symbols.

    Args:
    - start (str): Start date. (e.g., '2023-1-1 12:00:00+08:00')
    - freq (str): Frequency of data (e.g., '30m', '1d', '1h'). 
    - spot_li (list): List of symbols to download.
    - limit (int): Limit of data points to fetch.

    Example:
        download_spot_ohlcv('2023-1-1 12:00:00+08:00', '30m', ['BTCUSDT', 'ETHUSDT'])

    Mind
    - No symbol checking in symbol_li.
    - max_limit = 1000 for spot markets
    - timezone = 'UTC'
    """
    save_dir = os.path.join(os.path.dirname(__file__), 'data_base', self.exchange, self.symbol_type, freq)
    os.makedirs(save_dir, exist_ok=True)

    if not symbol_li:
        url_exchange_info = self.get_end_point() + self.get_suffix_exchange_info()
        exchange_info = requests.get(url_exchange_info).json()
        symbol_li = [i['symbol'] for i in exchange_info['symbols'] if i['quoteAsset']=='USDT']

    for symbol in symbol_li:
        symbol_amount = len(symbol_li)
        print(f"downloading {symbol_li.index(symbol)+1}/{symbol_amount}")

        df = self.fetch_ohlcv()



        df = self.fetch_spot_ohlcv(symbol, start, freq, limit)
        df['datetime'] = pd.to_datetime(df['datetime'], unit='ms').dt.tz_localize('UTC').dt.tz_convert(self.timezone)
        df = df.set_index(['datetime', 'code']).sort_index()

        file_name = f"{symbol}_ohlcv.pkl"
        file_path = os.path.join(save_dir, file_name)
        df.to_pickle(file_path)


        suffix = '/api/v3/exchangeInfo'
        url = self.end_point + suffix

        response = requests.get(url)
        exchange_info = response.json()
        
        spot_li = [i['symbol'] for i in exchange_info['symbols'] if i['quoteAsset']=='USDT']

    for symbol in spot_li:
        symbol_amount = len(spot_li)
        print(f"downloading {spot_li.index(symbol)+1}/{symbol_amount}")

        df = self.fetch_spot_ohlcv(symbol, start, freq, limit)
        df['datetime'] = pd.to_datetime(df['datetime'], unit='ms').dt.tz_localize('UTC').dt.tz_convert(self.timezone)
        df = df.set_index(['datetime', 'code']).sort_index()

        file_name = f"{symbol}_ohlcv.pkl"
        file_path = os.path.join(save_dir, file_name)
        df.to_pickle(file_path)

        print(f"{symbol} has been downloaded to {file_path}")


In [20]:
def fetch_spot_ohlcv(self, symbol, start, freq, limit=1000):
    """
    Mind
    - max_limit = 1000 for spot markets
    - timezone = 'UTC'
    """
    ohlcv_list = []
    start_ts13 = self.spot_client.parse8601(start)
    columns = ['datetime', 'open', 'high', 'low', 'close', 'volume']

    while True:
        ohlcv = self.spot_client.fetch_ohlcv(symbol, freq, since=start_ts13, limit=limit)
        ohlcv_list.extend(ohlcv)
        print(f"fetching {symbol} from {pd.to_datetime(start_ts13, unit='ms').tz_localize('UTC')}")
        if len(ohlcv) < limit:
            break    
        start_ts13 = ohlcv[-1][0]

    df = pd.DataFrame(ohlcv_list, columns=columns)
    df['code'] = symbol

    return df

IndentationError: unexpected indent (183207436.py, line 43)

In [ ]:
url = 'https://fapi.binance.com/fapi/v1/klines'
symbol = 'BTCUSDT'
interval = '1h'
start = (int(pd.to_datetime('2024-1-16').tz_localize('UTC').timestamp()*1000))
end = (int(pd.to_datetime('2024-1-17').tz_localize('UTC').timestamp()*1000))

par = {'symbol': symbol, 'interval': interval, 'startTime': start, 'limit':1500}
result = requests.get(url, params= par)

data = result.json()
df = pd.DataFrame(data, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_asset_volume', 'number_of_trades', 'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'ignore'])

# Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
df['close_time'] = pd.to_datetime(df['close_time'], unit='ms')



In [ ]:
df

,timestamp,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume,ignore
0,2024-01-16 00:00:00,42515.00,42679.00,42466.10,42604.70,4807.989,2024-01-16 00:59:59.999,204815547.27400,83161,2569.204,109445671.41200,0
1,2024-01-16 01:00:00,42604.60,42733.70,42556.60,42591.70,4728.842,2024-01-16 01:59:59.999,201602869.51377,77287,2468.202,105223777.86928,0
2,2024-01-16 02:00:00,42591.70,42732.80,42552.00,42702.10,5519.024,2024-01-16 02:59:59.999,235440131.60158,76101,3005.014,128184200.59768,0
3,2024-01-16 03:00:00,42702.10,42942.00,42695.70,42919.50,7032.131,2024-01-16 03:59:59.999,301186807.16910,95191,3731.711,159828199.19971,0
4,2024-01-16 04:00:00,42919.30,42945.00,42759.90,42805.20,5516.792,2024-01-16 04:59:59.999,236430231.24909,79812,2404.169,103019924.01450,0
...,...,...,...,...,...,...,...,...,...,...,...,...
148,2024-01-22 04:00:00,41199.50,41221.90,40755.50,40776.70,16886.142,2024-01-22 04:59:59.999,691938324.84133,197795,7244.088,296892497.02860,0
149,2024-01-22 05:00:00,40776.80,41153.70,40666.00,41133.60,16571.740,2024-01-22 05:59:59.999,678119430.09441,216322,9030.073,369521591.69400,0
150,2024-01-22 06:00:00,41133.60,41334.90,41002.00,41201.30,10932.641,2024-01-22 06:59:59.999,450049507.40093,140795,5694.183,234457663.32660,0
151,2024-01-22 07:00:00,41201.40,41210.30,40971.30,41042.60,6855.448,2024-01-22 07:59:59.999,281680490.23046,93563,2894.699,118953235.99750,0


In [ ]:
def fetch_ohlcv()

    par = {'symbol': symbol, 'interval': interval, 'startTime': start, 'endTime': end, 'limit':1500}
    result = requests.get(url, params= par)

    data = result.json()
    df = pd.DataFrame(data, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_asset_volume', 'number_of_trades', 'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'ignore'])

    # Convert timestamp to datetime
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df['close_time'] = pd.to_datetime(df['close_time'], unit='ms')


In [ ]:
start = '2024-1-16'

ohlcv_list = []
start_ts13 = (int(pd.to_datetime(start).tz_localize('UTC').timestamp()*1000))
columns = ['timestamp', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_asset_volume', 'number_of_trades', 'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'ignore']

symbol = ['']

while True:
    ohlcv = fetch_ohlcv(symbol, freq, since=start_ts13, limit=limit)
    ohlcv_list.extend(ohlcv)
    print(f"fetching {symbol} from {pd.to_datetime(start_ts13, unit='ms').tz_localize('UTC')}")
    if len(ohlcv) < limit:
        break    
    start_ts13 = ohlcv[-1][0]

df = pd.DataFrame(ohlcv_list, columns=columns)
df['code'] = symbol

In [ ]:
    def fetch_spot_ohlcv(self, symbol, start, freq, limit=1000):
        """
        Mind
        - max_limit = 1000 for spot markets
        - timezone = 'UTC'
        """
        ohlcv_list = []
        start_ts13 = self.spot_client.parse8601(start)
        columns = ['datetime', 'open', 'high', 'low', 'close', 'volume']

        while True:
            ohlcv = self.spot_client.fetch_ohlcv(symbol, freq, since=start_ts13, limit=limit)
            ohlcv_list.extend(ohlcv)
            print(f"fetching {symbol} from {pd.to_datetime(start_ts13, unit='ms').tz_localize('UTC')}")
            if len(ohlcv) < limit:
                break    
            start_ts13 = ohlcv[-1][0]

        df = pd.DataFrame(ohlcv_list, columns=columns)
        df['code'] = symbol

        return df

In [ ]:
df

,timestamp,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume,ignore
0,2024-01-15 16:00:00,42278.70,42321.70,42270.00,42293.90,195.416,2024-01-15 16:00:59.999,8265298.55050,2628,99.421,4205229.86100,0
1,2024-01-15 16:01:00,42293.90,42325.20,42249.10,42325.10,306.457,2024-01-15 16:01:59.999,12957396.44980,3671,132.720,5612473.52310,0
2,2024-01-15 16:02:00,42325.10,42329.00,42315.60,42326.00,141.747,2024-01-15 16:02:59.999,5999378.99640,2320,91.166,3858624.49430,0
3,2024-01-15 16:03:00,42325.90,42329.00,42312.70,42329.00,112.064,2024-01-15 16:03:59.999,4742941.73040,1497,41.884,1772775.89250,0
4,2024-01-15 16:04:00,42328.90,42329.00,42306.20,42317.30,118.162,2024-01-15 16:04:59.999,5000608.16390,1767,71.139,3010551.34960,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1495,2024-01-16 16:55:00,43223.30,43240.00,43191.10,43219.10,241.874,2024-01-16 16:55:59.999,10452382.17970,3232,77.441,3346566.98800,0
1496,2024-01-16 16:56:00,43219.00,43228.40,43188.00,43228.40,197.886,2024-01-16 16:56:59.999,8550386.59120,3121,113.102,4887004.80280,0
1497,2024-01-16 16:57:00,43228.40,43230.70,43190.00,43198.00,171.405,2024-01-16 16:57:59.999,7406332.77420,2907,54.102,2337457.31980,0
1498,2024-01-16 16:58:00,43198.10,43233.00,43198.10,43233.00,141.865,2024-01-16 16:58:59.999,6130617.98470,2076,94.103,4066084.44950,0


In [ ]:
df

,timestamp,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume,ignore
0,2024-01-16 00:00:00,42515.00,42679.00,42466.10,42604.70,4807.989,2024-01-16 00:59:59.999,204815547.27400,83161,2569.204,109445671.41200,0
1,2024-01-16 01:00:00,42604.60,42733.70,42556.60,42591.70,4728.842,2024-01-16 01:59:59.999,201602869.51377,77287,2468.202,105223777.86928,0
2,2024-01-16 02:00:00,42591.70,42732.80,42552.00,42702.10,5519.024,2024-01-16 02:59:59.999,235440131.60158,76101,3005.014,128184200.59768,0
3,2024-01-16 03:00:00,42702.10,42942.00,42695.70,42919.50,7032.131,2024-01-16 03:59:59.999,301186807.16910,95191,3731.711,159828199.19971,0
4,2024-01-16 04:00:00,42919.30,42945.00,42759.90,42805.20,5516.792,2024-01-16 04:59:59.999,236430231.24909,79812,2404.169,103019924.01450,0
5,2024-01-16 05:00:00,42805.20,42932.50,42759.00,42765.30,4148.775,2024-01-16 05:59:59.999,177768542.06161,62002,1804.142,77303801.58932,0
6,2024-01-16 06:00:00,42765.40,42779.10,42648.30,42744.40,5399.943,2024-01-16 06:59:59.999,230675732.23206,70080,2603.802,111232031.19860,0
7,2024-01-16 07:00:00,42744.50,42788.10,42600.50,42788.10,4965.136,2024-01-16 07:59:59.999,211981558.81910,72742,2339.181,99875735.40810,0
8,2024-01-16 08:00:00,42788.50,43029.70,42718.80,43021.50,8975.581,2024-01-16 08:59:59.999,385057362.00039,116370,4709.810,202042156.26929,0
9,2024-01-16 09:00:00,43021.50,43164.90,42899.60,42937.00,12361.569,2024-01-16 09:59:59.999,531854604.09884,154271,5766.724,248161235.10286,0


In [ ]:
result

<Response [404]>

In [ ]:
url = 'https://api.binance.com/api/v3/klines'
symbol = 'BTCUSDT'
interval = '1h'
start = (int(dt.datetime(2024,1,16).timestamp()*1000))
end = (int(dt.datetime(2024,1,17).timestamp()*1000))
par = {'symbol': symbol, 'interval': interval, 'startTime': start, 'endTime': end}
data = pd.DataFrame(json.loads(requests.get(url, params= par).text))
#format columns name
data.columns = ['datetime', 'open', 'high', 'low', 'close', 'volume','close_time', 'qav', 'num_trades','taker_base_vol', 'taker_quote_vol', 'ignore']
data.index = [dt.datetime.fromtimestamp(x/1000.0) for x in data.datetime]
data=data.astype(float)

In [ ]:
data

,datetime,open,high,low,close,volume,close_time,qav,num_trades,taker_base_vol,taker_quote_vol,ignore
2024-01-16 00:00:00,1.705334e+12,42278.99,42549.00,42249.00,42522.61,1121.26001,1.705338e+12,4.755025e+07,57874.0,627.94266,2.663062e+07,0.0
2024-01-16 01:00:00,1.705338e+12,42522.60,42779.78,42521.10,42696.58,1280.44041,1.705342e+12,5.463545e+07,60358.0,621.15357,2.650563e+07,0.0
2024-01-16 02:00:00,1.705342e+12,42696.57,43400.43,42696.57,43015.82,7891.28853,1.705345e+12,3.407552e+08,302218.0,3496.84512,1.508873e+08,0.0
2024-01-16 03:00:00,1.705345e+12,43015.82,43133.80,42825.84,42992.67,2038.31756,1.705349e+12,8.763074e+07,84396.0,996.74754,4.285295e+07,0.0
2024-01-16 04:00:00,1.705349e+12,42992.67,43114.45,42850.00,42960.00,1245.90158,1.705352e+12,5.357909e+07,51410.0,589.07810,2.533819e+07,0.0
2024-01-16 05:00:00,1.705352e+12,42959.99,42960.00,42710.00,42716.00,920.68651,1.705356e+12,3.942414e+07,42115.0,446.01729,1.909922e+07,0.0
2024-01-16 06:00:00,1.705356e+12,42716.00,42818.31,42342.18,42628.00,1850.23943,1.705360e+12,7.877147e+07,77507.0,872.71986,3.715815e+07,0.0
2024-01-16 07:00:00,1.705360e+12,42628.00,42639.99,42474.00,42511.10,1301.68577,1.705363e+12,5.539559e+07,52454.0,633.27021,2.695090e+07,0.0
2024-01-16 08:00:00,1.705363e+12,42511.10,42674.00,42466.88,42600.00,1007.25697,1.705367e+12,4.290120e+07,45056.0,549.01107,2.338284e+07,0.0
2024-01-16 09:00:00,1.705367e+12,42600.01,42726.97,42555.00,42584.34,875.50468,1.705370e+12,3.732198e+07,51073.0,481.01387,2.050440e+07,0.0


In [ ]:
requests.get?

Signature: requests.get(url, params=None, **kwargs)
Docstring:
Sends a GET request.

:param url: URL for the new :class:`Request` object.
:param params: (optional) Dictionary, list of tuples or bytes to send
    in the query string for the :class:`Request`.
:param \*\*kwargs: Optional arguments that ``request`` takes.
:return: :class:`Response <Response>` object
:rtype: requests.Response
File:      c:\users\hottari\anaconda3\envs\env3.9\lib\site-packages\requests\api.py
Type:      function

In [ ]:
start = int(pd.to_datetime('2024-1-16').timestamp()*1000)
end = int(pd.to_datetime('2024-1-17').timestamp()*1000)

In [ ]:
suffix = '/fapi/v1/klines'
url = end_point + suffix
params = {
    'symbol': 'BTCUSDT',
    'interval': '1T',
    'startTime': start, 
    'endTime': end,
    'limit': 1500
}

In [ ]:
response = requests.get(
    url, 
    params = params
)
#result = response.json()

In [ ]:
response.text

'<html>\r\n<head><title>404 Not Found</title></head>\r\n<body>\r\n<center><h1>404 Not Found</h1></center>\r\n<hr><center>nginx</center>\r\n</body>\r\n</html>\r\n'

In [ ]:
requests.get?

Signature: requests.get(url, params=None, **kwargs)
Docstring:
Sends a GET request.

:param url: URL for the new :class:`Request` object.
:param params: (optional) Dictionary, list of tuples or bytes to send
    in the query string for the :class:`Request`.
:param \*\*kwargs: Optional arguments that ``request`` takes.
:return: :class:`Response <Response>` object
:rtype: requests.Response
File:      c:\users\hottari\anaconda3\envs\env3.9\lib\site-packages\requests\api.py
Type:      function

In [ ]:
response

<Response [404]>